# D113 — Modules and Packages: MovieLens Example

A **module** is a `.py` file. A **package** is a directory of modules, normally containing `__init__.py`.

This lesson builds a small application inside `movielens-repo`, beside this notebook:

```text
movielens-repo/
`-- movielens/
    |-- __init__.py
    |-- __main__.py
    |-- config/
    |   `-- __init__.py
    |-- models/
    |   |-- __init__.py
    |   |-- movie.py
    |   `-- rating.py
    |-- datasets/
    |   |-- __init__.py
    |   |-- movie_dataset.py
    |   `-- rating_dataset.py
    |-- readers/
    |   |-- __init__.py
    |   |-- movie_reader.py
    |   `-- rating_reader.py
    `-- writers/
        |-- __init__.py
        `-- json_writer.py
```

Python module filenames use underscores, not hyphens: `movie_reader.py`, not `movie-reader.py`. The conventional spellings `writers` and `JsonWriter` are used.

Install the two external packages used by this example before running it:

```bat
python -m pip install simplejson python-dateutil
```

## 1. Create and use the repository directory

`%%writefile` writes relative paths from the current working directory. Jupyter normally starts the kernel in the directory containing the notebook. Create `movielens-repo` below that directory and use it as the project root.

In [ ]:
from pathlib import Path
import os

current_dir = Path.cwd()
notebook_dir = current_dir.parent if current_dir.name == "movielens-repo" else current_dir
lesson_root = notebook_dir / "movielens-repo"
lesson_root.mkdir(parents=True, exist_ok=True)
os.chdir(lesson_root)

for folder in (
    "movielens/models",
    "movielens/config",
    "movielens/datasets",
    "movielens/readers",
    "movielens/writers",
):
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())

## 2. Root package: `__init__.py`

`__init__.py` runs when `import movielens` executes. Keep it small; here it provides package documentation and a version.

In [ ]:
%%writefile movielens/__init__.py
"""MovieLens CSV loading example package."""

__version__ = "1.0.0"

## 3. Domain-model modules

Each model is an ordinary class. `__init__` stores its data and `__repr__` creates a readable display. No type hints are required.

In [ ]:
%%writefile movielens/models/movie.py
class Movie:
    def __init__(self, movie_id, title, genres):
        self.movie_id = movie_id
        self.title = title
        self.genres = genres

    def __repr__(self):
        return (
            f"Movie(movie_id={self.movie_id}, "
            f"title={self.title!r}, genres={self.genres!r})"
        )

In [ ]:
%%writefile movielens/models/rating.py
from datetime import datetime

from dateutil import tz


class Rating:
    def __init__(self, user_id, movie_id, rating, timestamp):
        self.user_id = user_id
        self.movie_id = movie_id
        self.rating = rating
        self.timestamp = timestamp

    @property
    def rated_at(self):
        return datetime.fromtimestamp(self.timestamp, tz=tz.UTC)

    def __repr__(self):
        return (
            f"Rating(user_id={self.user_id}, movie_id={self.movie_id}, "
            f"rating={self.rating}, timestamp={self.timestamp})"
        )

### Re-export the model classes

Relative imports begin with `.` and refer to the current package. Re-exporting gives callers a shorter public API:

```python
from movielens.models import Movie, Rating
```

`__all__` documents the intended public names.

In [ ]:
%%writefile movielens/models/__init__.py
"""Domain models exposed at the movielens.models package level."""

from .movie import Movie
from .rating import Rating

__all__ = ["Movie", "Rating"]

## 4. Dataset-container modules

A dataset owns a list of domain objects and provides small lookup operations. The movie dataset also builds an ID dictionary for fast lookup.

In [ ]:
%%writefile movielens/datasets/__init__.py
"""Dataset containers."""

In [ ]:
%%writefile movielens/datasets/movie_dataset.py
from movielens.models import Movie


class MovieDataset:
    def __init__(self, movies=None):
        self.movies = movies or []
        self._movies_by_id = {movie.movie_id: movie for movie in self.movies}

    def __len__(self):
        return len(self.movies)

    def __iter__(self):
        return iter(self.movies)

    def find_movie(self, movie_id):
        return self._movies_by_id.get(movie_id)

    def find_movie_by_title(self, title):
        search_text = title.casefold()
        return [movie for movie in self.movies if search_text in movie.title.casefold()]

In [ ]:
%%writefile movielens/datasets/rating_dataset.py
from movielens.models import Rating


class RatingDataset:
    def __init__(self, ratings=None):
        self.ratings = ratings or []

    def __len__(self):
        return len(self.ratings)

    def __iter__(self):
        return iter(self.ratings)

    def find_by_movie(self, movie_id):
        return [rating for rating in self.ratings if rating.movie_id == movie_id]

    def average_for_movie(self, movie_id):
        values = [rating.rating for rating in self.find_by_movie(movie_id)]
        return sum(values) / len(values) if values else None

`datasets/__init__.py` remains empty except for its docstring. Therefore import each class from its concrete module:

```python
from movielens.datasets.movie_dataset import MovieDataset
```

## 5. CSV-reader modules

The source data is:

- `C:\data\movielens\ml-latest-small\movies\movies.csv`
- `C:\data\movielens\ml-latest-small\ratings\ratings.csv`

Each reader converts CSV strings into domain objects and returns a dataset.

In [ ]:
%%writefile movielens/readers/__init__.py
"""CSV readers."""

In [ ]:
%%writefile movielens/readers/movie_reader.py
import csv
from pathlib import Path

from movielens.datasets.movie_dataset import MovieDataset
from movielens.models import Movie


class MovieReader:
    @staticmethod
    def read(csv_path):
        movies = []
        with Path(csv_path).open("r", encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                genres = tuple(row["genres"].split("|"))
                movies.append(Movie(int(row["movieId"]), row["title"], genres))
        return MovieDataset(movies)

In [ ]:
%%writefile movielens/readers/rating_reader.py
import csv
from pathlib import Path

from movielens.datasets.rating_dataset import RatingDataset
from movielens.models import Rating


class RatingReader:
    @staticmethod
    def read(csv_path):
        ratings = []
        with Path(csv_path).open("r", encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                ratings.append(
                    Rating(
                        user_id=int(row["userId"]),
                        movie_id=int(row["movieId"]),
                        rating=float(row["rating"]),
                        timestamp=int(row["timestamp"]),
                    )
                )
        return RatingDataset(ratings)

## 6. JSON-writer module

Each ordinary object stores its fields in `__dict__`. The writer uses the external `simplejson` package to serialize those dictionaries and creates `movielens-repo\output` when needed. The `Rating` model also uses the external `python-dateutil` distribution, imported as `dateutil`.

In [ ]:
%%writefile movielens/writers/__init__.py
"""Output writers."""

In [ ]:
%%writefile movielens/writers/json_writer.py
from pathlib import Path

import simplejson as json

from movielens.datasets.movie_dataset import MovieDataset
from movielens.datasets.rating_dataset import RatingDataset


class JsonWriter:
    @staticmethod
    def write_movies(dataset, output_path):
        return JsonWriter._write(dataset.movies, output_path)

    @staticmethod
    def write_ratings(dataset, output_path):
        return JsonWriter._write(dataset.ratings, output_path)

    @staticmethod
    def _write(items, output_path):
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as file:
            json.dump([item.__dict__ for item in items], file, indent=2)
        return path

Like datasets, writers are not re-exported from `writers/__init__.py`. Use:

```python
from movielens.writers.json_writer import JsonWriter
```

## 7. Central configuration module

File paths belong in one configuration module. Other modules import the values and do not need to know how configuration is obtained.

For now the values are defined directly in `config/__init__.py`. Later exercises can replace these definitions with environment variables or YAML without changing readers, writers, models, datasets, or the main workflow.

In [ ]:
%%writefile movielens/config/__init__.py
from pathlib import Path


DATA_ROOT = Path(r"C:\data\movielens\ml-latest-small")
OUTPUT_ROOT = Path(__file__).resolve().parents[2] / "output"

`Path(__file__).resolve().parents[2]` evaluates to `movielens-repo`: parent 0 is `config`, parent 1 is `movielens`, and parent 2 is the repository root.

## 8. Package entry point: `__main__.py`

`__main__.py` is different from `__init__.py`:

- `import movielens` executes `movielens/__init__.py`.
- `python -m movielens` executes `movielens/__main__.py` with package imports configured correctly.

The `if __name__ == "__main__"` guard calls `main()` only when this module is the program entry point.

In [ ]:
%%writefile movielens/__main__.py
import sys
from pathlib import Path

# Support direct execution: python movielens\__main__.py
if __package__ is None or __package__ == "":
    package_parent = Path(__file__).resolve().parent.parent
    sys.path.insert(0, str(package_parent))

from movielens.readers.movie_reader import MovieReader
from movielens.readers.rating_reader import RatingReader
from movielens.writers.json_writer import JsonWriter
from movielens.config import DATA_ROOT, OUTPUT_ROOT


def main():
    movies = MovieReader.read(DATA_ROOT / "movies" / "movies.csv")
    ratings = RatingReader.read(DATA_ROOT / "ratings" / "ratings.csv")

    print(f"Loaded {len(movies):,} movies")
    print(f"Loaded {len(ratings):,} ratings")

    toy_story = movies.find_movie(1)
    print("Movie 1:", toy_story)
    print("Title search:", movies.find_movie_by_title("Toy Story")[:3])
    print("Movie 1 average rating:", ratings.average_for_movie(1))

    movies_path = JsonWriter.write_movies(movies, OUTPUT_ROOT / "movies.json")
    ratings_path = JsonWriter.write_ratings(ratings, OUTPUT_ROOT / "ratings.json")
    print("Wrote:", movies_path)
    print("Wrote:", ratings_path)


if __name__ == "__main__":
    main()

## 9. Test imports inside the notebook

The lesson directory is the import root because it contains the `movielens` package directory.

In [ ]:
import sys

if str(lesson_root) not in sys.path:
    sys.path.insert(0, str(lesson_root))

from movielens.models import Movie, Rating
from movielens.readers.movie_reader import MovieReader
from movielens.readers.rating_reader import RatingReader

movies = MovieReader.read(
    r"C:\data\movielens\ml-latest-small\movies\movies.csv"
)
ratings = RatingReader.read(
    r"C:\data\movielens\ml-latest-small\ratings\ratings.csv"
)

print(type(movies.find_movie(1)))
print(type(ratings.ratings[0]))
print(f"Movies: {len(movies):,}; ratings: {len(ratings):,}")
print(movies.find_movie_by_title("Jumanji"))
print("Jumanji average:", ratings.average_for_movie(2))

## 10. Run the complete application

From Command Prompt, start in the directory that contains `movielens`:

```bat
C:\Users\GOPALAKRISHNANSUBRAM\dataeng\Scripts\activate.bat
cd /d movielens-repo
python -m movielens
dir output\*.json
```

Expected output files:

- `movielens-repo\output\movies.json`
- `movielens-repo\output\ratings.json`

The output path comes from `movielens.config`. It is calculated relative to `config\__init__.py`, so it does not depend on the current working directory.

Direct execution is also supported by this example:

```bat
python movielens\__main__.py
```

For direct execution, Python initially adds `movielens` itself to `sys.path`. The small bootstrap at the top adds `movielens-repo`, which is the directory containing the package. Prefer `python -m movielens` for normal package execution because Python configures this automatically.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "movielens"],
    cwd=lesson_root,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)

## 11. Import flow

```text
__main__.py
  ├─ config
  ├─ readers ──> models
  │      └─────> datasets ──> models
  └─ JsonWriter ──> datasets
```

Imports point from application code toward reusable domain code. The model modules do not import readers, datasets, writers, or the main program, which avoids circular imports.

### Summary

- Each `.py` file is a module.
- Each package directory contains `__init__.py`.
- `models/__init__.py` re-exports the public domain classes.
- `config/__init__.py` is the single location for input and output paths.
- Readers convert CSV rows into domain objects.
- Datasets hold objects and provide lookup methods.
- Writers serialize datasets.
- `python -m movielens` runs `movielens/__main__.py`.